# Date Range Discovery

Goal: find the min/max dates that appear across all claim date columns,
so we can pick safe `START_DATE` and `END_DATE` constants for the
`dim_date` calendar dimension.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data/raw')


def find_csv(prefix):
    """Tolerate Kaggle's timestamped filenames."""
    for cand in DATA_DIR.glob(f'{prefix}.csv'):
        return cand
    for cand in sorted(DATA_DIR.glob(f'{prefix}-*.csv')):
        return cand
    raise FileNotFoundError(prefix)


inp_dates = pd.read_csv(
    find_csv('Train_Inpatientdata'),
    usecols=['ClaimStartDt', 'ClaimEndDt', 'AdmissionDt', 'DischargeDt'],
    parse_dates=['ClaimStartDt', 'ClaimEndDt', 'AdmissionDt', 'DischargeDt'],
)

out_dates = pd.read_csv(
    find_csv('Train_Outpatientdata'),
    usecols=['ClaimStartDt', 'ClaimEndDt'],
    parse_dates=['ClaimStartDt', 'ClaimEndDt'],
)

print(f'Inpatient rows:  {len(inp_dates):>8,}')
print(f'Outpatient rows: {len(out_dates):>8,}')

Inpatient rows:    40,474
Outpatient rows:  517,737


In [2]:
all_dates = pd.concat([
    inp_dates['ClaimStartDt'], inp_dates['ClaimEndDt'],
    inp_dates['AdmissionDt'].dropna(), inp_dates['DischargeDt'].dropna(),
    out_dates['ClaimStartDt'], out_dates['ClaimEndDt'],
])

min_date = all_dates.min()
max_date = all_dates.max()
span_days = (max_date - min_date).days + 1

print(f'Earliest date:   {min_date.date()}')
print(f'Latest date:     {max_date.date()}')
print(f'Total span:      {span_days} days  ({span_days / 365.25:.2f} years)')
print()
print(f'Recommended dim_date constants (padded to whole years for safety):')
print(f'  START_DATE = "{min_date.year}-01-01"')
print(f'  END_DATE   = "{max_date.year}-12-31"')

Earliest date:   2008-11-27
Latest date:     2009-12-31
Total span:      400 days  (1.10 years)

Recommended dim_date constants (padded to whole years for safety):
  START_DATE = "2008-01-01"
  END_DATE   = "2009-12-31"
